# Lab 5: Text-to-SQL — Talk to Your Data
## CCI Session 3

**Duration:** 15 minutes

### Clinical Scenario
> The nursing director at KHCC wants to query patient vitals data without knowing SQL. You'll build a system where anyone can ask "How many patients had fever in the ICU this week?" and get an answer from the database.

### Objective
- Build the text-to-SQL pipeline: question → SQL → execute → answer
- Use few-shot examples to improve SQL generation
- Create a conversational interface with memory

---
### Setup
Run this cell first to install and import the required libraries.

In [ ]:
# Setup
!pip install openai pandas -q

import os
import pandas as pd
import sqlite3
from openai import OpenAI

# Set your API key
os.environ['OPENAI_API_KEY'] = 'your-api-key-here'  # Replace with your key
client = OpenAI()

print('Setup complete!')

---
## Cell 1 — Rebuild the Database

This cell rebuilds the database from Lab 4 so Lab 5 is self-contained.
Upload `vista_vitals.csv` and `vista_patients.csv` to Colab, then load them and create the SQLite database.

In [ ]:
# === CELL 1: SETUP DATABASE ===
# TODO: Load vista_vitals.csv and vista_patients.csv, create SQLite database
# This cell rebuilds the database so Lab 5 is self-contained

# df_vitals = pd.read_csv('vista_vitals.csv')
# df_patients = pd.read_csv('vista_patients.csv')
# conn = sqlite3.connect('khcc_vitals.db')
# df_vitals.to_sql('vista_vitals', conn, if_exists='replace', index=False)
# df_patients.to_sql('vista_patients', conn, if_exists='replace', index=False)

# TODO: Verify both tables
# pd.read_sql("SELECT COUNT(*) as total FROM vista_vitals", conn)
# pd.read_sql("SELECT COUNT(*) as total FROM vista_patients", conn)


---
## Cell 2 — Schema Description

The LLM needs to know the database structure to write correct SQL.
Create a detailed schema description string that includes:
- Both table names and all column names with types
- Sample values for each column
- Important notes about data quirks (like BP stored as text)
- The JOIN relationship between tables

In [ ]:
# === CELL 2: SCHEMA DESCRIPTION ===
# TODO: Create a string that describes the database schema for the LLM
# Include: table names, all column names with types, sample values for each column
# This is what you'll put in the system prompt

schema_description = """
Table: vista_vitals
Columns:
- NUMBER (integer): unique record ID
- MRN (text): encrypted patient medical record number  
- DATE_TIME_VITALS_TAKEN (text): when vitals were measured, format 'YYYY-MM-DDTHH:MM:SS.0000000'
- VITAL_TYPE (text): one of 'BLOOD PRESSURE', 'PULSE', 'PULSE OXIMETRY', 'RESPIRATION', 'TEMPERATURE'
- DATE_TIME_VITALS_ENTERED (text): when data was entered into the system
- HOSPITAL_LOCATION (text): ward name, e.g. 'KHCC-4C-HUSSAM AL-HARIRI', 'KHCC-ICU-CRITICAL CARE'
- ENTERED_BY (text): staff who recorded the vitals
- RATE (text): the vital sign value. For blood pressure it's 'systolic/diastolic' like '120/80'. For others it's a numeric string.

Table: vista_patients  
Columns:
- MRN (text): encrypted patient medical record number (JOIN key with vista_vitals)
- DATE_ESTABLISHED (text): when patient file was created
- DATE_OF_DEATH (text): '-' if alive, date string if deceased
- DOB (text): date of birth
- SEX (text): 'MALE' or 'FEMALE'
- NATIONALITY (text): 'JOR', 'PSE', 'IRQ', 'SYR', etc.
- PROVINCE (text): 'AMMAN', 'IRBID', 'ZARQA', etc.
- MARITAL_STATUS (text): 'MARRIED', 'SINGLE', 'WIDOWED', 'DIVORCED'
- NAME (text): Fernet-encrypted patient name
- PATIENT_TYPE (text): null, 'EARLY DETECTION', 'SCREENING', 'REFERRAL'

Relationship: vista_vitals.MRN = vista_patients.MRN (many vitals per patient)

Important notes:
- RATE is stored as TEXT, not a number. Use CAST(RATE AS FLOAT) for numeric comparisons (except blood pressure).
- Blood pressure RATE is in format 'systolic/diastolic' like '120/80'.
- Temperature is in Fahrenheit. Fever = temperature > 100.4.
- Normal pulse range: 60-100 bpm.
- Normal SpO2: >= 95%.
- Use JOINs when questions involve patient demographics (sex, nationality, province, age, etc.).
"""

print(schema_description)

# TODO: Also get sample rows to include in the prompt
# pd.read_sql("SELECT * FROM vista_vitals LIMIT 3", conn)
# pd.read_sql("SELECT * FROM vista_patients LIMIT 3", conn)


---
## Cell 3 — Basic Text-to-SQL

Write a function `text_to_sql(question)` that:
1. Sends the schema + question to GPT
2. Gets back a SQL query
3. Returns the SQL string

Your system prompt should include:
- The schema description
- Instruction to return ONLY the SQL query, nothing else
- Instruction to only generate SELECT queries (for safety!)

In [ ]:
# === CELL 3: TEXT TO SQL ===
# TODO: Write a function text_to_sql(question) that:
# 1. Sends the schema + question to GPT
# 2. Gets back a SQL query
# 3. Returns the SQL string

# System prompt should include:
# - The schema description
# - Instruction to return ONLY the SQL query, nothing else
# - Instruction to only generate SELECT queries (safety!)

def text_to_sql(question):
    pass  # TODO

# TODO: Test with: "How many total vital sign readings are in the database?"
# Print the generated SQL


---
## Cell 4 — Full Pipeline: Execute and Answer

Now build the complete pipeline. Write a function `ask_database(question)` that:
1. Calls `text_to_sql()` to get the SQL query
2. **Validates** that it's a SELECT query (safety check!)
3. Executes the SQL against the database
4. Sends the results back to GPT to formulate a natural language answer
5. Returns both the SQL and the human-readable answer

In [ ]:
# === CELL 4: FULL PIPELINE ===
# TODO: Write a function ask_database(question) that:
# 1. Calls text_to_sql() to get SQL
# 2. Validates it's a SELECT query (safety check!)
# 3. Executes the SQL against the database
# 4. Sends the results back to GPT to formulate a natural language answer
# 5. Returns both the SQL and the answer

def ask_database(question):
    pass  # TODO

# TODO: Test with these questions:
# "How many patients had fever in the ICU?"
# "Which ward has the most vital sign readings?"
# "What is the average pulse rate across all patients?"
# "How many male patients from Amman had high blood pressure?"
# "What's the average temperature for patients over 60?"
# "Show me vitals for Jordanian patients in the ICU"


---
## Cell 5 — Few-Shot Text-to-SQL

Few-shot examples dramatically improve SQL generation accuracy.
Add example question-to-SQL pairs to the system prompt and compare results.

**Examples to include:**
1. "How many unique patients are there?" → `SELECT COUNT(DISTINCT MRN) FROM vista_vitals`
2. "Show me all fever readings" → `SELECT * FROM vista_vitals WHERE VITAL_TYPE = 'TEMPERATURE' AND CAST(RATE AS FLOAT) > 100.4`
3. "Count readings per ward" → `SELECT HOSPITAL_LOCATION, COUNT(*) as count FROM vista_vitals GROUP BY HOSPITAL_LOCATION ORDER BY count DESC`
4. "How many female patients had fever?" → `SELECT COUNT(DISTINCT v.MRN) FROM vista_vitals v JOIN vista_patients p ON v.MRN = p.MRN WHERE v.VITAL_TYPE = 'TEMPERATURE' AND CAST(v.RATE AS FLOAT) > 100.4 AND p.SEX = 'FEMALE'`

In [ ]:
# === CELL 5: FEW-SHOT TEXT-TO-SQL ===
# TODO: Add example question→SQL pairs to the system prompt (including JOIN examples)
# Example 1: "How many unique patients are there?" → "SELECT COUNT(DISTINCT MRN) FROM vista_vitals"
# Example 2: "Show me all fever readings" → "SELECT * FROM vista_vitals WHERE VITAL_TYPE = 'TEMPERATURE' AND CAST(RATE AS FLOAT) > 100.4"
# Example 3: "Count readings per ward" → "SELECT HOSPITAL_LOCATION, COUNT(*) as count FROM vista_vitals GROUP BY HOSPITAL_LOCATION ORDER BY count DESC"
# Example 4 (JOIN): "How many female patients had fever?" → "SELECT COUNT(DISTINCT v.MRN) FROM vista_vitals v JOIN vista_patients p ON v.MRN = p.MRN WHERE v.VITAL_TYPE = 'TEMPERATURE' AND CAST(v.RATE AS FLOAT) > 100.4 AND p.SEX = 'FEMALE'"

# TODO: Update text_to_sql to use few-shot and test again
# Compare results with and without few-shot examples

# TODO: Test with JOIN-based questions:
# "How many male patients from Amman had high blood pressure?"
# "What's the average temperature for patients over 60?"
# "Show me vitals for Jordanian patients in the ICU"


---
## Cell 6 — Conversational Text-to-SQL

Combine text-to-SQL with conversation memory so the user can ask follow-up questions.
Build a chat loop where context carries over between questions.

**Test multi-turn conversation:**
1. "How many patients had fever?" → answer
2. "Which wards were they in?" → should use context from previous answer
3. "Show me the most recent fever reading" → continues the context

In [ ]:
# === CELL 6: CONVERSATIONAL TEXT-TO-SQL ===
# TODO: Combine text-to-SQL with conversation memory
# Build a chat loop where the user can ask follow-up questions

# conversation_messages = [system_message_with_schema]
#
# def chat(user_input):
#     # Add user message
#     # Get SQL from model
#     # Execute SQL
#     # Get NL answer
#     # Add assistant response to history
#     # Return answer
#     pass

# TODO: Test multi-turn:
# "How many patients had fever?" → answer
# "Which wards were they in?" → should use context from previous answer
# "Show me the most recent fever reading" → continues the context


---
## Cell 7 — Text-to-SQL with Tool Calling

Instead of generating SQL directly in text, use **tool calling** (from Lesson 3).
Define a tool `run_sql_query(sql: str) -> str` that the model can call.
The model generates the SQL as a tool call argument, you execute it and return results.

This is cleaner because the model explicitly "asks" to query the database
rather than mixing SQL into its text response.

In [ ]:
# === CELL 7: TEXT-TO-SQL WITH TOOL CALLING ===
# TODO: Instead of generating SQL directly, use tool calling
# Define a tool: run_sql_query(sql: str) -> str
# The model generates the SQL as a tool call argument
# You execute it and return results
# This is cleaner because the model explicitly "asks" to query the database

# This combines Lesson 3 (tool calling) with Lesson 5 (text-to-SQL)!


---
## Stretch Challenge

Add **error handling** to your pipeline:
- If the generated SQL fails to execute, catch the error
- Send the error message back to the model
- Let the model retry with a corrected query
- Limit retries to 3 attempts

In [ ]:
# === STRETCH CHALLENGE: ERROR HANDLING ===
# Your code here


---
### KHCC Connection

This is the foundation for the **natural language clinical data query system** being planned
for KHCC ward dashboards. Imagine a nurse asking "Show me all ICU patients with unstable
vitals in the last 4 hours" and getting an instant answer — no SQL knowledge required.
The text-to-SQL pipeline you built here is the core of that system.